## Using the trained model to predict in our dataset

### First we import the packages and the dataset

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

import pandas as pd

# Start by importing the test data
test_path = Path('data/raw/dataset_test.csv')
df_test = pd.read_csv(test_path)

Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Now we perform the datacleaning

In [2]:
from src.data_cleaning import clean_data

df_test = clean_data(dataset= df_test, is_train = False, categorize_station=True)

df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 537445 entries, 0 to 537444
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   id                    537445 non-null  str           
 1   datetime              537445 non-null  datetime64[us]
 2   station_number        537445 non-null  category      
 3   name                  537445 non-null  str           
 4   lat                   537445 non-null  float64       
 5   lng                   537445 non-null  float64       
 6   hour                  537445 non-null  int32         
 7   minute                537445 non-null  int32         
 8   dayofweek             537445 non-null  int32         
 9   month                 537445 non-null  int32         
 10  is_weekend            537445 non-null  int64         
 11  hour_sin              537445 non-null  float64       
 12  hour_cos              537445 non-null  float64       
 13  is_holiday

In [3]:
# Save IDs and do preprocessing (as we did before)

test_ids = df_test["id"].copy()      # save the ids

DROP_COLS = ["id", "datetime", "name"]#,"lat", "lng" 

df_test = df_test.drop(columns = DROP_COLS)


In [5]:
df_test

,station_number,lat,lng,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos,is_holiday,temperature_2m,apparent_temperature,precipitation,snowfall,wind_speed_10m,cloud_cover,relative_humidity_2m
0,32000,48.211544,16.382374,8,30,3,3,0,0.866025,-0.500000,0,10.1,7.2,0.5,0.0,16.4,100,88
1,32001,48.210666,16.372983,8,30,3,3,0,0.866025,-0.500000,0,10.1,7.2,0.5,0.0,16.4,100,88
2,32002,48.202683,16.369702,8,30,3,3,0,0.866025,-0.500000,0,10.1,7.2,0.5,0.0,16.4,100,88
3,32003,48.206516,16.360400,8,30,3,3,0,0.866025,-0.500000,0,10.1,7.2,0.5,0.0,16.4,100,88
4,32004,48.219522,16.382218,8,30,3,3,0,0.866025,-0.500000,0,10.1,7.2,0.5,0.0,16.4,100,88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537440,32275,48.250224,16.450650,23,30,1,4,0,-0.258819,0.965926,0,14.6,13.3,0.0,0.0,5.6,0,66
537441,32277,48.227914,16.391516,23,30,1,4,0,-0.258819,0.965926,0,14.6,13.3,0.0,0.0,5.6,0,66
537442,32278,48.224598,16.392090,23,30,1,4,0,-0.258819,0.965926,0,14.6,13.3,0.0,0.0,5.6,0,66
537443,32280,48.251355,16.452810,23,30,1,4,0,-0.258819,0.965926,0,14.6,13.3,0.0,0.0,5.6,0,66


### Do the predictions

In [9]:
import joblib
import pandas as pd
import numpy as np

# Load the trained model
model_path = Path('models/lightgbmv2.joblib')
model = joblib.load(model_path)

# Make predictions
predictions = model.predict(df_test)

#rund the predictions
predictions = np.maximum(0, predictions).round().astype(int)


### Prepare the submission

In [11]:
submission = pd.DataFrame({"id": test_ids, "bikes": predictions})

# and now save the predictions
from pathlib import Path

out_path = Path('reports/predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)